In [1]:
# cerca il modulo ml dalle cartelle sys.path partendo dalla cartella superiore:

from pathlib import Path
import sys

current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (path / "ml" / "dataset.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Non trovo ml/dataset.py. Controlla la cartella di lavoro."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Percorso aggiunto a sys.path:", str(PROJECT_ROOT) in sys.path)

Project root: C:\Users\ale\Machine_learning\AWS_intro\fullstack-ml-app-on-aws
Percorso aggiunto a sys.path: True


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ml.dataset import df_analysis, X, y
from ml.feature_config import numeric_features, binary_features, categorical_features
from ml.preprocessing import make_preprocessor
from ml.data_split import X_train, X_test, y_train, y_test

# Intro dei dati preprocessati importati

In [ ]:

print("=== CHECK IMPORTED X,y DATA ===")

print("\n--- X ---")
print(X.shape)

print("\n--- y ---")
print(y.shape)

print("\n--- Feature utilizzate ---")
print(X.columns.tolist()) # crea un list/array delle colonne di X. Id e' stato tolto correttamente

print("\n--- Target ---")
print(y.name)

# Pulizia e tipizzazione delle feature
Partiamo da X, non da df_analysis, perché abbiamo già separato target e identificatore.

In [ ]:


print("\n--- Data types prima della conversione ---")
print(X.dtypes)

print("\n--- Missing values prima della conversione ---")
print(X.isna().sum().sort_values(ascending=False))




# Separazione delle feature numeriche e categoriche


In [ ]:
print("=== CHECK IMPORTED FEATURE GROUPS ===")

print("Numeric features:")
print(numeric_features)

print("\nBinary features:")
print(binary_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumero feature numeriche:", len(numeric_features))
print("Numero feature binarie:", len(binary_features))
print("Numero feature categoriche:", len(categorical_features))   


# Encoding del target

In [ ]:
print("=== CHECK IMPORTED TARGET DATA ENCODED ===")

print("\n--- Target dopo la conversione ---")
print(y.value_counts())

print("\n--- Target dtype ---")
print(y.dtype)

print("\n--- Valori mancanti ---")
print(y.isna().sum())

# Importiamo il ColumnTransformer (Fondamentale per la Pipeline che andremo a creare)


In [ ]:
# La funzione make_preprocessor e'definita nel file preprocessing.py
preprocessor = make_preprocessor(numeric_features, binary_features, categorical_features) # preprocessing.py import

print(preprocessor)

print("=== CHECK PREPROCESSOR  ===")
print("\n--- Feature totali ---")
print(len(numeric_features) + len(binary_features) + len(categorical_features)) # 19

#

print("\n--- Preprocessor configurato ---")
print(preprocessor.transformers) ## "mostra la configurazione dei trasformatori che ho definito."



# Check Train/Test Split e Fit e trasformazione delle feature


In [ ]:
print("CHECK IMPORTED TRAIN / TEST SPLIT ===")

print("\n--- Shapes ---")
print("X_train", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\n--- Target distribution TRAIN ---")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True)*100)

print("\n--- Target distribution TEST ---")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True) * 100)


# Eseguiamo il fit e la trasformazione dei dati per verificare che il preprocessor funziona correttamente 

(Essenziale da verificare prima di usare la pipeline nel Cross-validation)


In [ ]:
print("=== FIT & TRANSFORM di train e test data ===")

X_train_processed = preprocessor.fit_transform(X_train) # one hot encoder applied in preprocessing.py 
X_test_processed = preprocessor.transform(X_test)


print("\n--- Shapes dopo preprocessing ---")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)
# 45 colonne delle x features dopo transformazione encoding




# Trasformiamo l'array numpy in dataframe pandas solo per mostrare la trasformazione dell encoder sulle variabili categortiche¶


In [ ]:
# Trasformiamo l'array numpy in dataframe pandas solo per mostrare la trasformazione dell encoder sulle variabili categortiche¶
feature_names = preprocessor.get_feature_names_out() # get column

X_train_processed_df = pd.DataFrame( # transform numpy array in Pandas datafraeme
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_train_processed_df

# Verifica del preprocessing


In [ ]:
import numpy as np

print("\n--- Shape ---")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)

print("\n--- Numero feature ---")
print(len(feature_names))

print("\n--- NaN nel training set ---")
print(np.isnan(X_train_processed).sum())

print("\n--- NaN nel test set ---")
print(np.isnan(X_test_processed).sum())

print("\n--- Valori unici nelle prime 10 feature ---")
for i, feature in enumerate(feature_names[:10]):
    print(feature, "→", np.unique(X_train_processed[:, i]))


# PIPELINE, CROSS-VALIDATION e OOF
Creiamo la pipeline dove inseriamo il ColumnTransformer creato in precedenza, in modo da non avere **Data leakage** durante il cross-validation k-fold

In [ ]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report
) 

In [ ]:
preprocessor = make_preprocessor(numeric_features, binary_features, categorical_features)


from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate
from scipy.stats import randint # permette di definire intervalli di valori interi da esplorare,
# anziché elencare ogni valore possibile.


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

from sklearn.base import clone # non modifichiamo ripetutamente la pipeline, creiamo invece una nuova pipeline per ogni configurazione, riutilizzando il preprocessor.
# clone(preprocessor), fa' in modo che il preprocessor sia riutilizzabile, facendo 'dimenticare' al preprocessor i precedenti fit.transform del train set. 
# Quindi possiamo riutilizzare il preprocessor senza correre rischi di data leakage. 


xgb_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", XGBClassifier(
        objective="binary:logistic", #→ classificazione binaria(yes - no) con output probabilistico predict_proba(per misurare auc score)
        eval_metric="logloss", # metrica coerente con una classificazione binaria probabilistica
        random_state=42,
        n_jobs=1
    ))
])

print("Pipeline XGBoost creata correttamente.")
print(xgb_pipeline)


# Definiamo la ricerca degli iperparametri con RandomizedSearchCV
Dopodiche' eseguiamo un ulteriore **cross-validate** tuning manuale su alcuni iperparametri
Usiamo queste strategie cross validations per scegliere gli iperparametri perche' cosi' facendo valutiamo le metriche direttamente sul **train set**, **lasciando il test set "indipendente" per il test finale**.


In [ ]:
# Definiamo gli intervalli di ricercha degli iperparametri
from scipy.stats import randint, uniform, loguniform

xgb_param_distributions = {
    "classifier__n_estimators": randint(100, 501),
    "classifier__max_depth": randint(2, 8),
    "classifier__learning_rate": loguniform(0.01, 0.2),
    "classifier__min_child_weight": randint(1, 11),
    "classifier__subsample": uniform(0.6, 0.4),
    "classifier__colsample_bytree": uniform(0.6, 0.4),
    "classifier__gamma": uniform(0, 2)
}

In [ ]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline, # Per ogni k-fold, trasformiamo le colonne categoriche con OneHotEncoder, usiamo il modello per il traning(sui gruppi di train) e valuations(sul gruppo di test)
    param_distributions=xgb_param_distributions, # gli intervalli degli iperparametri che vogliamo analizzare per ottenere il miglior f1 score, come stabilito in "scoring"
    n_iter=30, # Per ogni k-fold, RandomizedSearchCV proverà 30 combinazioni di iperparametri selezionate dallo spazio di ricerca definito, e su richiesta ritorna i parametri che hanno portato al "best" f1 score 
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
    verbose=1,
    return_train_score=False
)

xgb_search.fit(X_train,y_train)


print("Migliori parametri:")
print(xgb_search.best_params_)

print("\nMiglior F1 medio in cross-validation:")
print(round(xgb_search.best_score_, 4))


# Ulteriore tuning "manuale" su alcuni parametri, lasciando fissi quelli ottenuti con RandomizedSearchCV:
Il tuning verra' sempre fatto con cross-validate



In [ ]:
# Parametri migliori trovati da RandomizedSearchCV
best_params = xgb_search.best_params_.copy()

# Copiamo i "migliori parametri per f1" ottenuti da RandomizedSearchCV per poi copiarli nella cross-validate.
xgb_params = {
    key.replace("classifier__", ""): value
    for key, value in best_params.items()
}

# controlliamo se i parametri sono corretti
print(f"check params: {xgb_params} ")


In [ ]:
# min_child_weight: 
child_weight_values = [1, 3, 5, 7, 10, 12]
results_xgb_manual = []

for child_weight in child_weight_values:

    params = xgb_params.copy()
    params["min_child_weight"] = child_weight

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1,
        **params # i params scelti tramite RandomSearchCv + il tuning di min_child_weight
    )

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", model)
    ])

# Eseguiamo validazione oof con gli iperparametri trovati con RandomizedSearchCv, e facendo ulteriore tuning manuale su child_weight
    scores = cross_validate( 
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "f1": "f1",
            "precision": "precision",
            "recall": "recall"
        },
        n_jobs=-1
    )

    results_xgb_manual.append({
        "min_child_weight": child_weight,
        "F1 medio": scores["test_f1"].mean(),
        "F1 std": scores["test_f1"].std(),
        "Precision media": scores["test_precision"].mean(),
        "Recall medio": scores["test_recall"].mean()
    })

In [ ]:
# vediamo i risultati 
results_xgb_manual_df = pd.DataFrame(results_xgb_manual)

results_xgb_manual_df = results_xgb_manual_df.sort_values(
    by="F1 medio",
    ascending=False
).reset_index(drop=True)

print(results_xgb_manual_df.round(4).to_string(index=False))

# Come suggerito da RandomizedSearchCv, min_child_weight=1 rimane la configurazione migliore

# Consolidiamo e addestriamo il modello finale

Utilizziamo direttamente xgb*search.best_estimator*, che contiene già la pipeline con il preprocessing, gli parametri migliori per f1 score, ed è **già stato riaddestrato su tutto X_train grazie a parametro refit=True**.


In [ ]:
# Modello XGBoost consolidato grazie a refit=True. xgb_search.best_estimator_ contiene l'intera pipeline + i best_params ottenuti da RandomizeSearchCv.
xgb_final = xgb_search.best_estimator_

print("Parametri del modello finale:")
print(xgb_final.named_steps["classifier"].get_params())

print("\nModello XGBoost consolidato correttamente.")

In [ ]:
print(xgb_final)

# 2. Calcoliamo metriche e confusion matrix

In [ ]:
y_pred_xgb = xgb_final.predict(X_test) # La pipeline si occupa delle trasformazioni delle colonne di X_test 

# Probabilità della classe positiva: Churn = 1
y_proba_xgb = xgb_final.predict_proba(X_test)[:,1] # possibile grazie alla metrica xgboost = objective="binary:logistic"

# Metriche
metrics_xgb = {
    "Accuracy": accuracy_score(y_test,y_pred_xgb),
    "Precision": precision_score(y_test, y_pred_xgb, zero_division=0),
    "Recall": recall_score(y_test, y_pred_xgb, zero_division=0),
    "F1-score": f1_score(y_test, y_pred_xgb, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test,y_proba_xgb)
}

metrics_xgb_df = pd.DataFrame(
    metrics_xgb.items(),
    columns=["Metrica", "Valore"]
)

print("METRICHE XGBOOST")
print(metrics_xgb_df.round(4).to_string(index=False))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test,
    y_pred_xgb,
    target_names=["No churn", "Churn"],
    zero_division=0
))

# XGBoost vince ancora e si candida come modello favorito sopra Random Forest. Semplicemente perche' raggiunge un F1 più alto e una
# precision maggiore

# Find Best threshold:

Cerchiamo la soglia decisionale ideale per aumentare i TP al minor costo possibile di FP.

Scegliamo la soglia usando le predizioni **out-of-fold (OOF)**

Generiamo le probabilità out-of-fold

In [ ]:
from sklearn.model_selection import cross_val_predict

# Ogni osservazione del training set viene predetta
# da un modello che non ha visto quell'osservazione
oof_proba = cross_val_predict(
    clone(xgb_final), # clone() crea una copia non addestrata del modello, mantenendo gli iperparametri selezionati. # cross_val_predict avvia addestramento del modello vergine clonato sui fold previsti dalla cross-validation.
    X_train,
    y_train,
    cv=cv,
    method="predict_proba", # Valutazione con predizioni OOF
    n_jobs=-1
)[:, 1] # ritorna le probs dei churn = 1

print("Predizioni out-of-fold generate:", len(oof_proba)) # lunghezza dei dati di training. Corrisponde con in nr di rows del traning set


# Tuning delle le soglie. 
Vediamo quale di queste soglie offre miglior trade-off tra **precision e recall**


In [ ]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (oof_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_train,
        y_pred_threshold,
        labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Soglia": threshold,
        "Recall": recall_score(y_train, y_pred_threshold),
        "Precision": precision_score(
            y_train, y_pred_threshold, zero_division=0
        ),
        "F1": f1_score(y_train, y_pred_threshold),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))

# La soglia 0.35 sembra offrire il miglior F1 tra le soglie testate e recupera più clienti a rischio rispetto
# a 0.50, contenendo parte dei falsi positivi rispetto alle soglie più basse.

# Scegliamo una di queste soglie con la seguente Condizione:
# Recall almeno del 70%, con il minor numero di FP

In [ ]:
threshold_candidates = threshold_df[
    threshold_df["Recall"] >= 0.70
].sort_values(
    by=["FP", "Soglia"],
    ascending=[True, False]
)

print(threshold_candidates.round(4).to_string(index=False))

# La soglia 0.35 risulta ancora la migliore secondo la specifica

# Applichiamo la soglia favorita al test set:

In [ ]:
# Probabilità di churn sul test set
y_test_proba = xgb_final.predict_proba(X_test)[:, 1]

# Soglia selezionata tramite probs OOF
threshold = 0.35
y_test_pred_035 = (y_test_proba >= threshold).astype(int) # True=1

# Metriche
print("Soglia:", threshold)
print("Accuracy:", round(accuracy_score(y_test, y_test_pred_035), 4))
print("Precision:", round(precision_score(y_test, y_test_pred_035), 4))
print("Recall:", round(recall_score(y_test, y_test_pred_035), 4))
print("F1-score:", round(f1_score(y_test, y_test_pred_035), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_test_proba), 4))

print("\nMatrice di confusione:")
print(confusion_matrix(y_test, y_test_pred_035))

print("\nClassification report:")
print(classification_report(y_test, y_test_pred_035))



# Confronto dei modelli

In [ ]:
model_comparison = pd.DataFrame([
    {
        "Modello": "Random Forest",
        "Soglia": 0.50,
        "Accuracy": 0.7956,
        "Precision": 0.6378,
        "Recall": 0.5321,
        "F1-score": 0.5802,
        "ROC-AUC": 0.8392
    },
    {
        "Modello": "XGBoost",
        "Soglia": 0.50,
        "Accuracy": 0.8048,
        "Precision": 0.6667,
        "Recall": 0.5294,
        "F1-score": 0.5902,
        "ROC-AUC": 0.8472
    },
    {
        "Modello": "XGBoost",
        "Soglia": 0.35,
        "Accuracy": 0.7800,
        "Precision": 0.5684,
        "Recall": 0.7112,
        "F1-score": 0.6318,
        "ROC-AUC": 0.8472
    }
])

print(model_comparison.round(4).to_string(index=False))

# Fine tuning e valutazione metriche di XGBoost.

Le prestazioni di XGBoost vincono su quelle di Random Forest al momento.
Da mensionare che i parametri class_weight="balanced"**(RF)** e scale_pos_weigh**(XGB)** non sono stati testati, da testare in futuro cosi' come abbiamo fatto con gli altri iperparametri:

es:

scale_pos_weights = [1.0, 1.5, 2.0, 2.8]

weight_results = []

for weight in scale_pos_weights:
     model = clone(xgb_search.best_estimator_)

     model.set_params(
         classifier__scale_pos_weight=weight
     )
    scores = cross_validate(
         model,
         X_train,
         y_train,
         cv=cv,
         ...............

# Analisi interpretativa del modello finale con Permutation Importance e SHAP.

Iniziamo valutando le feature singolarmente. 
Successivamente, raggrupperemo quelle che risultano associate secondo i
calcoli di correlazione/associazione definiti nel file **inspect_dataset.py**

Per la nostra **Permutation Importance**, usiamo il modello finale xgb_final con soglia 0.35.

# **Passaggio 1 — Calcoliamo la Permutation Importance**


In [ ]:
from sklearn.inspection import permutation_importance


# Soglia selezionata tramite OOF validation (la soglia migliore per il nostro modello finale che risulta come miglior f1 score)
threshold = 0.35

# Funzione per calcolare l'F1 usando la soglia personalizzata. 
# Per ogni feature, mescoleremo i valori 20 volte e misuriamo quanto cambia l'F1-score.

def f1_at_threshold(estimator, X, y):
    probabilities = estimator.predict_proba(X)[:, 1] # probs di churn=1 del modello candidato xgb_final 
    predictions = (probabilities >= threshold).astype(int)
    return f1_score(y, predictions, zero_division=0) # zero_division=0 evita error in casi in cui non vengano predetti positivi

# Permutation Importance sul modello XGBoost tuned
perm_importance_xgb = permutation_importance(
    estimator=xgb_final,
    X=X_test,
    y=y_test,
    scoring=f1_at_threshold, # Passiamo a f1_at_threshold() gli argomenti di permutation_importance (xgb_final, X(le features da valutare X_test originali e poi mescolate), y(i dati y_test per la validazione))
    n_repeats=20,  # mescola i dati di ogni feature 20 volte e calcola f1 
    random_state=42,
    n_jobs=-1
)

print("Permutation Importance calcolata correttamente.")
print("Numero di feature analizzate:", len(perm_importance_xgb.importances_mean))


# Passaggio 2 — Visualizziamo le feature più importanti

In [ ]:

# Creiamo una tabella con i risultati
perm_importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm_importance_xgb.importances_mean, # media dei valori mescolati 20 volte per features
    "Importance_Std": perm_importance_xgb.importances_std # devisazione standard
})

# Ordiniamo dalla più importante alla meno importante
perm_importance_df = perm_importance_df.sort_values(
    by="Importance_Mean",
    ascending=False
).reset_index(drop=True)

# Stampiamo la classifica completa
# display(perm_importance_df.round(4))
print(perm_importance_df.round(4))

Secondo questa prima analisi individuale, **Contract e' nettamente la features piu' importante** per la previsione del target.
Riguardo alle features associate, risulta invece che tenure e InternetService sono le piu' importanti


# Visualizziamo le prime 15 feature 

In [ ]:
top_15 = perm_importance_df.head(15).sort_values(
    by="Importance_Mean",
    ascending=True
)

plt.figure(figsize=(10, 7))
plt.barh(
    top_15["Feature"],
    top_15["Importance_Mean"],
    xerr=top_15["Importance_Std"]
)
plt.axvline(x=0, linestyle="--")
plt.xlabel("Diminuzione media dell'F1-score")
plt.ylabel("Feature")
plt.title("Permutation Importance — XGBoost tuned (soglia 0.35)")
plt.tight_layout()
plt.show()



**I dati della Permutation importance possono risultare inaffidabili se una o piu' features sono correlate**. Se una features spiega una altra, questo metodo non e' il migliore, e va' fatto solo come una prima analisi esplorativa.
Per risolvere questo problema, su applica la **Permutazione per gruppi**, ovvero si raggruppano le features correlate, si mescolano i dati di questo gruppo di feature(in modo da spezzare la relazione con le altre features variabili e target) e si misura quanto le metrica f1 cambia rispetto al modello originale. Lo stesso workflow della permutation fatta sulle singole features, ma si applica su un gruppo di 2 o piu' features correlate.

# Permutazione per gruppi. 
Ragruppiamo le features associate calcolate nel file inspect_dataset.py, e raggruppiamole per misurare nuovamente la Permutation_importance


# Analiziamo il primo gruppo di features associate:

- tenure
- TotalCharges

In [ ]:
# Gruppo di feature associate
group_features = ["tenure", "TotalCharges"]

# F1 di riferimento, senza permutazioni
baseline_f1 = f1_at_threshold(xgb_final, X_test, y_test)

group_importances = []

rng = np.random.default_rng(42) # crea un generatore di numeri casuali di NumPy.

for _ in range(20):
    X_permuted = X_test.copy()

    # Stessa permutazione delle righe per entrambe le feature:
    # la relazione interna tra tenure e TotalCharges viene preservata
    permutation = rng.permutation(len(X_permuted)) # indici mesolati delle righe del train set

    X_permuted[group_features] = ( # usa gli indici shuffled per mescolare i dati di group_features, mantenendo la relazione tra esse ma spezzandole con il resto delle features
        X_test[group_features].iloc[permutation].to_numpy()
    ) # 

    permuted_f1 = f1_at_threshold( # ripetiamo 20 volde lo shuffle, e la valutazione delle metriche del modello finale sul test set
        xgb_final,
        X_permuted,
        y_test
    )
    # Quanto peggiora l'F1 rispetto al riferimento
    group_importances.append(baseline_f1 - permuted_f1)

group_importance_mean = np.mean(group_importances)
group_importance_std = np.std(group_importances, ddof=0)

print(f"F1-score di riferimento: {baseline_f1:.4f}")
print(f"Importanza congiunta media: {group_importance_mean:.4f}")
print(f"Deviazione standard: {group_importance_std:.4f}")

La permutazione congiunta di tenure e TotalCharges provoca una diminuzione dell'F1-score considerevole.
Questo indica che il gruppo contiene informazioni predittive importanti per il tuo modello.


# Analizzare il secondo gruppo di features associate:

- PhoneService
- MultipleLines

In [ ]:
# Gruppo di feature associate
group_features = ["PhoneService", "MultipleLines"]

# F1 di riferimento
baseline_f1 = f1_at_threshold(xgb_final, X_test, y_test)

group_importances_phone = []

# rng = np.random.default_rng(42)

for _ in range(20):
    X_permuted = X_test.copy()

    # Stessa permutazione per entrambe le feature
    permutation = rng.permutation(len(X_permuted))

    X_permuted[group_features] = (
        X_test[group_features].iloc[permutation].to_numpy()
    )

    permuted_f1 = f1_at_threshold(
        xgb_final,
        X_permuted,
        y_test
    )

    group_importances_phone.append(
        baseline_f1 - permuted_f1
    )

group_importance_mean_phone = np.mean(group_importances_phone)
group_importance_std_phone = np.std(
    group_importances_phone,
    ddof=0
)

print(f"F1-score di riferimento: {baseline_f1:.4f}")
print(
    "Importanza congiunta media: "
    f"{group_importance_mean_phone:.4f}"
)
print(
    "Deviazione standard: "
    f"{group_importance_std_phone:.4f}"
)


PhoneService e MultipleLines mostrano un'importanza congiunta piuttosto bassa, nonostante avere un'associazione forte secondo il calcolo **Chi-square-test e cramer**

# Analisi del terzo gruppo di features associate:


In [ ]:
# Gruppo di servizi Internet e servizi aggiuntivi
group_features_internet = [
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

# F1 di riferimento
baseline_f1 = f1_at_threshold(xgb_final, X_test, y_test)

group_importances_internet = []

# rng = np.random.default_rng(42)

for _ in range(20):
    X_permuted = X_test.copy()

    # Applichiamo la stessa permutazione a tutte le feature del gruppo
    permutation = rng.permutation(len(X_permuted))

    X_permuted[group_features_internet] = (
        X_test[group_features_internet]
        .iloc[permutation]
        .to_numpy()
    )

    permuted_f1 = f1_at_threshold(
        xgb_final,
        X_permuted,
        y_test
    )

    group_importances_internet.append(
        baseline_f1 - permuted_f1
    )

group_importance_mean_internet = np.mean(
    group_importances_internet
)
group_importance_std_internet = np.std(
    group_importances_internet,
    ddof=0
)

print(f"F1-score di riferimento: {baseline_f1:.4f}")
print(
    "Importanza congiunta media: "
    f"{group_importance_mean_internet:.4f}"
)
print(
    "Deviazione standard: "
    f"{group_importance_std_internet:.4f}"
)



Il terzo gruppo ha un’importanza congiunta media di 0 1127, quindi la permutazione simultanea dei servizi Internet provoca un calo significativo dell’F1-score. 

Cio' indica che il modello perde una parte importante della sua capacità predittiva quando gli togliamo l'associazione originale tra i clienti e il loro gruppo di servizi. 

# Analisi del quarto gruppo di features associate:

- Contract 
- tenure

In [ ]:
# Gruppo: contratto e durata del cliente
group_features_contract = ["Contract", "tenure"]

# F1 di riferimento
baseline_f1 = f1_at_threshold(xgb_final, X_test, y_test)

group_importances_contract = []
# rng = np.random.default_rng(42)

for _ in range(20):
    X_permuted = X_test.copy()

    # Stessa permutazione per entrambe le feature
    permutation = rng.permutation(len(X_permuted))

    X_permuted[group_features_contract] = (
        X_test[group_features_contract]
        .iloc[permutation]
        .to_numpy()
    )

    permuted_f1 = f1_at_threshold(
        xgb_final,
        X_permuted,
        y_test
    )

    group_importances_contract.append(
        baseline_f1 - permuted_f1
    )

group_importance_mean_contract = np.mean(
    group_importances_contract
)
group_importance_std_contract = np.std(
    group_importances_contract,
    ddof=0
)

print(f"F1-score di riferimento: {baseline_f1:.4f}")
print(
    "Importanza congiunta media: "
    f"{group_importance_mean_contract:.4f}"
)
print(
    "Deviazione standard: "
    f"{group_importance_std_contract:.4f}"
)

Questo è il risultato più alto tra i gruppi che abbiamo analizzato finora
Il modello passa da un F1-score di 0.6318 a circa 0.4215 quando permutiamo insieme Contract e tenure. 
Quindi il gruppo contiene informazioni predittive molto importanti per il modello.


# SHAP
SHAP ci aiuterà a capire quali feature spingono le previsioni verso il churn e quali nella direzione opposta.


# Prepariamo i dati per SHAP

Per ottenere spiegazioni leggibili, cercheremo di mantenere i nomi delle feature originali, come Contract e InternetService, invece di ritrovarci soltanto con le numerose colonne create dall'One-Hot Encoding

# Verifichiamo il preprocessing:


In [ ]:
# Estraiamo il preprocessing e il classificatore dalla pipeline
preprocessor = xgb_final.named_steps["preprocessor"]
xgb_classifier = xgb_final.named_steps["classifier"]

# Trasformiamo i dati di test nello stesso modo usato dal modello
X_test_transformed = preprocessor.transform(X_test)

# Recuperiamo i nomi delle feature dopo il preprocessing
feature_names_transformed = preprocessor.get_feature_names_out()

print("Forma di X_test originale:", X_test.shape)
print("Forma di X_test trasformato:", X_test_transformed.shape)
print("Numero di nomi delle feature:", len(feature_names_transformed))

print("\nPrimi 15 nomi delle feature trasformate:")
print(feature_names_transformed[:15])

# Le 19 feature originali sono diventate 45 feature numeriche dopo il preprocessing. 
# SHAP deve analizzare i dati trasformati, perché sono quelli che XGBoost riceve effettivamente


# Prepariamo il campione di training

Per iniziare, useremo un campione casuale di 500 clienti del training set. È un buon compromesso iniziale tra velocità di calcolo e quantità di informazioni.


In [ ]:
# Campione del training set per l'analisi SHAP
X_shap = X_train.sample(
    n=500,
    random_state=42
)

# Applichiamo lo stesso preprocessing usato dal modello
X_shap_transformed = preprocessor.transform(X_shap)

print("Forma del campione originale:", X_shap.shape)
print("Forma del campione trasformato:", X_shap_transformed.shape)

# Il campione è pronto


# Calcoliamo i valori SHAP

In [ ]:
import shap

# Creiamo lo spiegatore SHAP per XGBoost
explainer = shap.TreeExplainer(xgb_classifier)

# Calcoliamo i valori SHAP sul campione trasformato
shap_values = explainer(X_shap_transformed)

print("Forma dei valori SHAP:", shap_values.values.shape)
print("Numero di feature trasformate:", len(feature_names_transformed))


# Calcoliamo l'importanza globale SHAP
Per ogni feature calcoliamo la media del valore assoluto dei suoi valori SHAP. In questo modo misuriamo quanto la feature contribuisce alle previsioni, senza confondere contributi positivi e negativi che potrebbero annullarsi.


In [ ]:
import pandas as pd
import numpy as np

# Importanza globale: media dei valori SHAP assoluti
shap_importance = pd.DataFrame({
    "Feature": feature_names_transformed,
    "Mean_Abs_SHAP": np.abs(shap_values.values).mean(axis=0)
})

# Ordiniamo dalla più importante alla meno importante
shap_importance = shap_importance.sort_values(
    "Mean_Abs_SHAP",
    ascending=False
).reset_index(drop=True)

print(shap_importance.head(20).to_string(index=False))


**Contract_Month-to-month** ha il contributo medio più grande alle previsioni del modello.

**tenure** è al secondo posto: la durata del rapporto con il cliente influenza sensibilmente le previsioni.


# Ora mostriamo la direzione dell'effetto delle features values


In [ ]:
shap.summary_plot(
    shap_values.values,
    X_shap_transformed,
    feature_names=feature_names_transformed,
    max_display=20
)


# il grafico mostra che:
**Contract_Month-to-month** risulta essere ancora una volta la feature piu' importante per il target. 

I punti rossi (che corrispondono a **OHE** = 1 della categoria) prevalentemente a destra della retta indicano che avere un contratto mensile spinge il modello verso una maggiore probabilità di churn.

Poi viene tenure, OnlineSecurity_N, etc...

# Prepariamo l'importanza SHAP per le 19 feature originali

Creiamo un grafico aggregato sulla base dei dati precedenti.

L'idea è raggruppare le colonne derivate dalla stessa variabile originale (cioe' le colonne categoriche trasformate da One-Hot_encoding). 
Per esempio:

**Contract_Month-to-month, Contract_One year e Contract_Two year** appartengono tutte a **Contract**. 

Per ogni osservazione, sommeremo i valori SHAP con il loro segno all'interno di ciascun gruppo; poi calcoleremo la media dei valori assoluti.



# Aggregazione SHAP dalle colonne preprocessate alle feature originali


In [ ]:
# Matrice dei valori SHAP: righe = clienti, colonne = feature preprocessate
shap_df = pd.DataFrame(
    shap_values.values,
    columns=feature_names_transformed,
    index=X_shap.index
)

# Associa ogni colonna trasformata alla feature originale (19 colonne)
original_features = {}

for col in feature_names_transformed:
    if col.startswith("numeric__"):
        original_features[col] = col.replace("numeric__", "")
    elif col.startswith("binary__"):
        original_features[col] = col.replace("binary__", "")
    elif col.startswith("categorical__"):
        # Rimuove il prefisso e la categoria, conservando il nome originale
        transformed_name = col.replace("categorical__", "")
        original_features[col] = next(
            feature
            for feature in categorical_features
            if transformed_name.startswith(feature + "_")
        )
    else:
        raise ValueError(f"Prefisso non riconosciuto: {col}")


# Somma i contributi SHAP con segno per ciascuna feature originale


In [ ]:
shap_original = pd.DataFrame(index=shap_df.index)

for feature in X_train.columns:
    columns = [
        col for col in shap_df.columns
        if original_features[col] == feature
    ]
    shap_original[feature] = shap_df[columns].sum(axis=1)


# Importanza globale: media del valore assoluto del contributo aggregato


In [ ]:
shap_original_importance = (
    shap_original.abs()
    .mean()
    .sort_values(ascending=False)
    .rename("Mean_Abs_SHAP")
    .reset_index()
    .rename(columns={"index": "Feature"})
)

print("Numero di feature originali:", shap_original.shape[1])
display(shap_original_importance)

# Risultato di **SHAP aggregato** molto simile a quello di shap sulle features preprocessate con One-hot-Encoder:


# Creiamo il grafico di SHAP agregata


In [ ]:
# Ordina le feature dalla meno alla più importante
plot_data = shap_original_importance.sort_values(
    "Mean_Abs_SHAP", ascending=True
)

plt.figure(figsize=(10, 8))

plt.barh(
    plot_data["Feature"],
    plot_data["Mean_Abs_SHAP"]
)

plt.xlabel("Mean Absolute SHAP Value (aggregated)")
plt.ylabel("Original Feature")
plt.title("SHAP Feature Importance — Original Features")
plt.tight_layout()
plt.show()


# salvare la pipeline

Salviamo la pipeline finale per il **deployment su AWS**

In [ ]:
from pathlib import Path
import joblib
import json

In [ ]:
# Directory del modello
MODEL_DIR = Path("artifacts")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(MODEL_DIR)

In [ ]:
# Salva pipeline completa: preprocessing + XGBoost
joblib.dump(xgb_final, MODEL_DIR / "xgb_final.joblib")

# Salva la soglia decisionale
with open(MODEL_DIR / "threshold.json", "w", encoding="utf-8") as f:
    json.dump({"threshold": 0.35}, f, indent=2)

print("Modello e soglia salvati correttamente.")

# Testiamo il modello salvato prima di passare al **deployment su AWS**


In [ ]:
# Valutiamo se il "loaded-model" e' stato salvato correttamente e e confrontiamo le probabilità con quelle prodotte dall'oggetto originale su alcune righe di test.
loaded_model = joblib.load("artifacts/xgb_final.joblib")

original_proba = xgb_final.predict_proba(X_test.iloc[:10])[:, 1]
loaded_proba = loaded_model.predict_proba(X_test.iloc[:10])[:, 1]

import numpy as np

np.testing.assert_allclose(
    original_proba,
    loaded_proba,
    rtol=1e-6,
    atol=1e-7
)

print("Verifica superata: probabilità coerenti.")

# probabilità coerenti. Modello idoneo per deployment su AWS!


# Creiamo inference.py 
Realizziamo una funzione Python **indipendente da AWS**, così possiamo verificarla localmente prima di introdurre il formato richiesto dal **container SageMaker**


In [ ]:
# Testiamo una predizione su una riga del test set originale.

sample_customer = X_test.iloc[0].to_dict()
print(sample_customer)


In [ ]:
from ml.inference import predict_customer

result = predict_customer(sample_customer)
print(result)

# Test di inferenza locale superato. 

# Passiamo al training del modello su Sagemaker.
- Creiamo il file train.py con le istruzioni di allenamento per aws sagemaker.
- Installiamo SageMaker sdk e boto3
- Verifichiamo che gli ambienti Sagemaker sono compatibili con le dipendenze installate in locale
- Procediamo con allenamento modello su cloud e salvataggio degli artefatti su s3

In [1]:
# import sys
# import sagemaker
# import boto3

# print("Python:", sys.executable)
# print("SageMaker:", sagemaker)
# print("Boto3:", boto3.__version__)


# Verifichiamo che Scikit-learn 1.9-0 sia disponibile

In [ ]:
from sagemaker.core import image_uris

# Versioni Scikit-learn disponibili
versions = image_uris.list_versions(
    framework="sklearn",
    image_scope="training"
)

print("Versioni disponibili:")
print(versions)

Versioni disponibili:
['0.20.0', '0.23-1', '1.0-1', '1.2-1', '1.4-2', '1.9-0', '1.4-2-py312']


# Verifichiamo anche la versione Python
È importante che ambiente locale e quello SageMaker utilizzino le stesse versioni Python 3.11.3.

In [11]:
python_versions = image_uris.list_py_versions(
    framework="sklearn",
    version="1.9-0",
    image_scope="training"
)

print("Versioni Python supportate:")
print(python_versions)

Versioni Python supportate:
['py3']


# Recuperiamo l'immagine Docker di Francoforte

In [12]:
image_uri = image_uris.retrieve(
    framework="sklearn",
    region="eu-central-1",
    version="1.9-0",
    image_scope="training",
    instance_type="ml.m5.large"
)

print("Container Scikit-learn:")
print(image_uri)

Container Scikit-learn:
492215442770.dkr.ecr.eu-central-1.amazonaws.com/sagemaker-scikit-learn:1.9-0-cpu-py3
